---
image: example.gif
pub-info:
    abstract: |
        A gas station model where each fuel pump/resource has its own continuously changing status -
        the tank's fuel level - shown alongside the entity icons. Demonstrates visualising a more
        complex, numeric resource state rather than just 'busy' or 'free'.
execute: 
  enabled: true
---

# Visualising a more complex status alongside entity icons - gas station with individual fuel tank level

::: {.callout-tip}
## Adding synchronised charts to an animation

The bottom half of this notebook builds a second, synchronised chart panel underneath the
animation. Since vidigi 2.0.0 there are helpers for this - `vidigi.animation.add_subplot_panels`,
`add_synchronised_trace` and `add_synchronised_trace_from_dataframe` - which handle the fiddly
parts (keeping the extra trace in step with every frame without losing the stage labels or
resource icons). This example uses them. See
[Synchronised traces](../feat_synchronised_traces/index.ipynb) for a focused walkthrough.
:::

In [1]:
from vidigi.utils import EventPosition, create_event_position_df
from vidigi.prep import reshape_for_animations, generate_animation_df
from vidigi.animation import (
    generate_animation,
    animate_activity_log,
    add_subplot_panels,
    add_synchronised_trace_from_dataframe,
)
import pandas as pd
import os
import random
import plotly.io as pio
import plotly.graph_objects as go
import plotly.express as px

# "iframe" keeps the executed notebook small (the heavy Plotly HTML is written to a
# gitignored iframe_figures/ folder instead of embedded inline). Use "notebook" if you
# are running this interactively and want the animation inline.
pio.renderers.default = "iframe"

First we bring in our code. 

This example is a modified version of the gas station example created by team SimPy. Updates have been done to incorporate vidigi resources to allow for resource tracking. 

In [2]:
#| echo: false
#| output: asis
# Path to the external Python script
file_path = "simpy_gas_stations.py"

# Read the file content
if os.path.exists(file_path):
    with open(file_path, "r") as f:
        code_content = f.read()
else:
    code_content = "File not found."
with open(file_path, "r") as f:
    code_content = f.read()

# Print the Quarto `{details}` block for collapsible output
print(f"""
:::{{.callout-note collapse="true"}}
### View Imported Code, which has had logging steps added at the appropriate points in the 'model' class

```python
{code_content}
```

:::

""")


:::{.callout-note collapse="true"}
### View Imported Code, which has had logging steps added at the appropriate points in the 'model' class

```python
"""
Gas Station Refueling example

Covers:

- Resources: Resource
- Resources: Container
- Waiting for other processes

Scenario:
  A gas station has a limited number of gas pumps that share a common
  fuel reservoir. Cars randomly arrive at the gas station, request one
  of the fuel pumps and start refueling from that reservoir.

  A gas station control process observes the gas station's fuel level
  and calls a tank truck for refueling if the station's level drops
  below a threshold.

  simpy model (simpy_gas_stations.py) written by Team SimPy as part of SimPy documentation.

"""

import itertools
import random

from vidigi.resources import VidigiStore
from vidigi.animation import animate_activity_log
from vidigi.logging import EventLogger
from vidigi.utils import EventPosition, create_event_position_df

import simpy

# fmt: off
RAND

First, we build our animation in the normal way. 

In [3]:
# Define positions for animation
event_positions = create_event_position_df([
    EventPosition(event='arrival', x=0, y=350, label="Entrance"),
    EventPosition(event='pump_queue_wait_begins', x=400, y=350, label="Queue"),
    EventPosition(event='payment_begins', x=340, y=175, resource='num_pumps',
                  label="Pumping Gas"),
    EventPosition(event='pumping_begins', x=340, y=175, resource='num_pumps',
                  label="Pumping Gas"),
    EventPosition(event='calling_truck', x=140, y=50,
                  label="Calling Truck"),
        EventPosition(event='refuelling', x=340, y=50,
                  label="Truck Filling Tank"),
    EventPosition(event='depart', x=250, y=50, label="Exit")
])

class Params:
    def __init__(self):
        self.num_pumps = 2

icon_list = [ "🚗", "🚙", "🚓",
            "🚗", "🚙", "🏍️", "🏍️",
            "🚗", "🚙", "🚑",
            "🚗", "🚙", "🛻",
            "🚗", "🚙", "🚛",
            "🚗", "🚙", "🚕",
            "🚗", "🚙", "🚒",
            "🚗", "🚙", "🚑"]

random.shuffle(icon_list)

In [4]:
event_log_df = pd.read_csv("gas_station_log.csv")

In [5]:
STEP_SNAPSHOT_MAX = 6
LIMIT_DURATION = 60*60*3
WRAP_QUEUES_AT = 3

In [6]:
full_entity_df = reshape_for_animations(
    event_log=event_log_df,
    every_x_time_units=5,
    step_snapshot_max=STEP_SNAPSHOT_MAX,
    limit_duration=LIMIT_DURATION,
    debug_mode=True
    )

full_entity_df_plus_pos = generate_animation_df(
    full_entity_df=full_entity_df,
    event_position_df=event_positions,
    wrap_queues_at=WRAP_QUEUES_AT,
    step_snapshot_max=STEP_SNAPSHOT_MAX,
    gap_between_entities=150,
    gap_between_resources=180,
    gap_between_queue_rows=150,
    # gap_between_resource_rows=60,
    debug_mode=True,
    custom_entity_icon_list=icon_list
    )


C:\Users\Sammi\AppData\Local\Temp\ipykernel_30660\2598734781.py:1: UserWarning:

2 entities ('parameter', 'StationTank') have events in the event log but no 'arrival' event, so they will be missing from every frame of the animation.

vidigi works out who is present at each snapshot from the arrival and departure rows, so an entity without an arrival is never drawn.

The usual cause is discarding a warm-up period by filtering the log, e.g. `event_log[event_log['time'] >= warm_up]`, which removes the arrival rows of everyone already in the system - including entities that are still queuing.

To skip a warm-up period, pass the whole event log and set `warm_up` to the end of the warm-up instead. That trims the animation window without discarding the history it needs.



Iteration through time-unit-by-time-unit logs complete 11:04:48


Snapshot df concatenation complete at 11:04:48
Placement dataframe started construction at 11:04:48
Placement dataframe finished construction at 11:04:48


Let's now define a custom function that uses the amount of fuel to generate a bar for each individual car that can update as they fill up. 

In [7]:
def build_fuel_bar(value, max_value=50, length=10):
    """Create an ASCII bar to show fuel level."""
    try:
        if value is None or (isinstance(value, float) and (value != value)):  # check for None or NaN
            proportion = 0
        else:
            proportion = min(max(value / max_value, 0), 1)
    except Exception:
        proportion = 0  # fallback

    filled = int(proportion * length)
    empty = length - filled
    filled_icon = "█"
    empty_icon = "░"
    return "[" + filled_icon * filled + empty_icon * empty + "]"


Now we can combine this with a function that will apply a custom icon to different kinds of entities - the refill trucks, the action of calling the truck (both of which we'll display as a string of text plus an icon), and the custom icons that reflect the stage of transaction individual entities are at, along with their fuel level at that point. 

In [8]:
def custom_icon_rules(row):
    icon = row.get("icon", "")
    entity_id = row.get("entity_id", "")
    event = row.get("event", "")
    fuel_level_start = row.get("fuel_level_start", None)  # Only for cars

    if "more" not in str(icon):
        if isinstance(entity_id, str):
            if "Truck" in entity_id:
                return "🚚 Truck is refilling the tank..."
            elif "Call" in entity_id:
                return "☎️ Calling Truck!"
            elif "Car" in entity_id:
                bar = ""
                if (event == "arrival" or event == "pump_queue_wait_begins") and fuel_level_start is not None:
                    bar = " " + build_fuel_bar(fuel_level_start)
                    return icon + "<br>" + bar + "<br><br>"
                elif event == "payment_begins" and fuel_level_start is not None:
                    bar = " " + build_fuel_bar(fuel_level_start)
                    return icon+ "<br>" + bar + "<br> Paying"
                elif event == "pumping_begins" and fuel_level_start is not None:
                    arrival_time = row["time"]
                    elapsed = max(float(row["snapshot_time"]) - float(arrival_time), 0)
                    current_fuel = min(fuel_level_start + elapsed * 1, 50)
                    bar = " " + build_fuel_bar(current_fuel)
                    return icon+ "<br>" + bar + "<br> Pumping"
                elif event == "departure" or event == "pumping_ends":
                    bar = " " + build_fuel_bar(50)  # Car is full when it leaves
                    return icon+ "<br>" + bar + "<br> <br>"


            else:
                return icon
    return icon


full_entity_df_plus_pos = full_entity_df_plus_pos.assign(
            icon=full_entity_df_plus_pos.apply(custom_icon_rules, axis=1)
            )

Finally we create our animation. Note that instead of including a resource icon in the normal way, we've added the resources as part of our background. This can be a more visually pleasing option when the number of resources is something that isn't going to change. 

In [9]:
fig = generate_animation(
        full_entity_df_plus_pos=full_entity_df_plus_pos.sort_values(['entity_id', 'snapshot_time']),
        event_position_df= event_positions,
        scenario=Params(),
        simulation_time_unit="seconds",
        plotly_height=900,
        plotly_width=1200,
        override_x_max=500,
        override_y_max=750,
        entity_icon_size=30,
        gap_between_resources=180,
        display_stage_labels=False,
        # resource_opacity=1,
        resource_opacity=0,
        setup_mode=False,
        # custom_resource_icon="⛽",
        resource_icon_size=40,
        add_background_image="https://raw.githubusercontent.com/hsma-tools/vidigi/refs/heads/main/examples/example_15_gas_station_refuelling/gas_station.png",
        background_image_opacity=1, # New parameter in 1.1.0
        overflow_text_color="white", # New parameter in 1.1.0
        start_time="09:00:00",
        time_display_units="%H:%M:%S",
        debug_mode=True,
        frame_duration=100,
        frame_transition_duration=100
    )

fig

Output animation generation complete at 11:04:59


Now let's explore visualising the total amount of fuel available in the station's tank. 

In [10]:
fuel_level_change_df = event_log_df[(event_log_df["event_type"]=="fuel_level_change") &
                                    (event_log_df["time"] % 5 == 0) &
                                    (event_log_df["time"] < LIMIT_DURATION)]

px.bar(fuel_level_change_df, x="entity_id", y="value", animation_frame="time", range_y=[0,400])

Next, we can incorporate the tank fuel level as an additional synchronised chart beneath the
animation, using `add_subplot_panels` and `add_synchronised_trace_from_dataframe`.

We'll first regenerate the animation, with a little more height to leave room for the second
panel.

In [11]:
## Same as before, but increase the height to give space for some of it to be taken up by the bar plot later

fig = generate_animation(
        full_entity_df_plus_pos=full_entity_df_plus_pos.sort_values(['entity_id', 'snapshot_time']),
        event_position_df= event_positions,
        scenario=Params(),
        simulation_time_unit="seconds",
        plotly_height=1000,
        plotly_width=1200,
        override_x_max=500,
        override_y_max=750,
        entity_icon_size=30,
        gap_between_resources=180,
        display_stage_labels=False,
        # resource_opacity=1,
        resource_opacity=0,
        setup_mode=False,
        # custom_resource_icon="⛽",
        resource_icon_size=40,
        add_background_image="https://raw.githubusercontent.com/hsma-tools/vidigi/refs/heads/main/examples/example_15_gas_station_refuelling/gas_station.png",
        background_image_opacity=1, # New parameter in 1.1.0
        overflow_text_color="white", # New parameter in 1.1.0
        start_time="09:00:00",
        time_display_units="%H:%M:%S",
        debug_mode=True,
        frame_duration=100,
        frame_transition_duration=100
    )

Output animation generation complete at 11:05:14


### Step 1: make room for the second panel

`add_subplot_panels` turns the single-axis animation into a two-row subplot grid, with the
animation in the top row. `row_heights` sets how the vertical space is split. We keep the new
panel's axes visible (`hide_new_panel_axes=False`) so the fuel scale can be read.

In [12]:
fig = add_subplot_panels(
    fig,
    row_heights=[0.78, 0.22],
    subplot_titles=("", "Station Tank Fuel Level"),
    hide_new_panel_axes=False,
)

In [13]:
# One tank reading per animation snapshot (forward-filled between fuel_level_change events)
tank_level = (
    event_log_df.loc[event_log_df["event_type"] == "fuel_level_change", ["time", "value"]]
    .drop_duplicates("time")
    .set_index("time")["value"]
)

station_fuel_df = pd.DataFrame(
    {"snapshot_time": sorted(full_entity_df_plus_pos["snapshot_time"].unique())}
)
station_fuel_df["fuel_level"] = (
    station_fuel_df["snapshot_time"].map(tank_level).ffill().bfill()
)


def fuel_bar(rows):
    return go.Bar(
        x=["Station Tank"],
        y=list(rows["fuel_level"]),
        marker_color="#4c78a8",
        showlegend=False,
        xaxis="x2",
        yaxis="y2",
    )


fig = add_synchronised_trace_from_dataframe(
    fig,
    station_fuel_df,
    fuel_bar,
    frame_time_col="snapshot_time",
    match="index",
    accumulate=False,
)

fig.update_yaxes(range=[0, station_fuel_df["fuel_level"].max() * 1.1], row=2, col=1)

### Step 2: add the synchronised bar

The `fuel_level_change` events record the tank level once per simulated second. We take one
reading per animation snapshot - the same `snapshot_time` values `generate_animation_df`
produced - so there is exactly one row per frame, then let
`add_synchronised_trace_from_dataframe` build a bar for each.

`match="index"` pairs the i-th row with the i-th frame, so it doesn't matter that the frames
are labelled as clock times while our data is in seconds. `accumulate=False` means each frame
sees only its own row (a snapshot, not a running total).

We now have a complex animation showing a lot of different information for entities and the
situation - and the tank fuel level panel stays in step across every frame, not just the first.

Drag the slider and you can watch the tank draining as cars fill up, then jumping back up
each time the delivery truck refills it.

In [14]:
fig